# Практика 4. Сравниваем четыре классификатора

**Финансовый университет · Технологии финансовой аналитики**

Определим, какие операции похожи на мошенничество. Создадим пять новых признаков,
обучим логистическую регрессию, дерево, случайный лес и градиентный бустинг.
В конце соберём их метрики в одну таблицу.

Положите `card_operations.csv` рядом с ноутбуком. В Colab загрузите его в панель «Файлы».
Запускайте ячейки сверху вниз.

## 1. Загружаем данные

В файле **5 000 операций**. Одна строка — одна операция.
`target = 1` означает фрод, `target = 0` — обычную покупку.
Классы уже известны по результатам разбирательства; модель будет учиться предсказывать их по признакам операции.

| Исходный признак | Что означает |
|---|---|
| `amount_rub` | Сумма текущей покупки, ₽ |
| `avg_amount_30d` | Средняя сумма прошлых покупок клиента за 30 дней, ₽ |
| `operations_1h` | Число предыдущих операций клиента за последний час |
| `operations_24h` | Число предыдущих операций за последние 24 часа |
| `hour` | Час текущей покупки: от 0 до 23 |
| `new_device` | Новое устройство: 1 — да, 0 — нет |
| `distance_km` | Расстояние от привычного места покупок, км |

Это **семь исходных признаков**. `operation_id` — идентификатор, `target` — ответ; их не включаем в признаки.
Все исторические показатели доступны до текущего запроса на покупку. `avg_amount_30d` положителен,
пропусков в файле нет.

In [1]:
import pandas as pd

data = pd.read_csv("card_operations.csv")
data.head()

,operation_id,amount_rub,avg_amount_30d,operations_1h,operations_24h,hour,new_device,distance_km,target
0,T00001,7594.63,3931.81,0,6,0,0,8.8,0
1,T00002,639.18,1824.99,0,5,11,0,9.9,0
2,T00003,1792.22,1828.59,4,5,18,0,11.4,1
3,T00004,6051.13,3182.84,1,8,14,0,16.4,0
4,T00005,729.62,2175.43,1,9,23,1,4.2,0


In [2]:
print("Операций:", len(data))
print("Фродов:", data["target"].sum())
print("Доля фрода:", round(data["target"].mean() * 100, 1), "%")

Операций: 5000
Фродов: 686
Доля фрода: 13.7 %


## 2. Добавляем пять признаков

Новый признак — ещё один способ описать ту же операцию. Например, сумма 10 000 ₽ сама по себе
мало говорит: для одного клиента она обычная, а для другого в десять раз выше привычной.

| Новый признак | Как считаем |
|---|---|
| `amount_ratio` | Текущая сумма / средняя сумма прошлых покупок |
| `is_night` | Покупка с 00:00 до 05:59 — 1, иначе 0 |
| `hour_activity_share` | Операции за час / (операции за сутки + 1) |
| `new_device_large_amount` | Новое устройство **и** сумма больше 10 000 ₽ — 1, иначе 0 |
| `far_from_usual` | Расстояние больше 100 км — 1, иначе 0 |

`.astype(int)` превращает `True` / `False` в 1 / 0.
В доле операций прибавляем 1 к знаменателю, чтобы не делить на ноль.
Границы 10 000 ₽ и 100 км фиксируем заранее.

In [3]:
data["amount_ratio"] = data["amount_rub"] / data["avg_amount_30d"]
data["is_night"] = (data["hour"] <= 5).astype(int)
data["hour_activity_share"] = data["operations_1h"] / (data["operations_24h"] + 1)
data["new_device_large_amount"] = ((data["new_device"] == 1) & (data["amount_rub"] > 10000)).astype(int)
data["far_from_usual"] = (data["distance_km"] > 100).astype(int)

data[["amount_rub", "amount_ratio", "is_night", "hour_activity_share",
      "new_device_large_amount", "far_from_usual"]].head()

,amount_rub,amount_ratio,is_night,hour_activity_share,new_device_large_amount,far_from_usual
0,7594.63,1.931586,1,0.000000,0,0
1,639.18,0.350238,0,0.000000,0,0
2,1792.22,0.980110,0,0.666667,0,0
3,6051.13,1.901173,0,0.111111,0,0
4,729.62,0.335391,0,0.100000,0,0


## 3. Отделяем признаки и откладываем операции для сравнения

В **X** включаем все **12 признаков: 7 исходных + 5 новых**. В **y** — столбец `target`.

Оставим 75% операций для обучения, 25% — для сравнения моделей (**validation**).
Все четыре модели будут проверяться на одних и тех же операциях.

In [4]:
from sklearn.model_selection import train_test_split

base_features = ["amount_rub", "avg_amount_30d", "operations_1h", "operations_24h",
                 "hour", "new_device", "distance_km"]
new_features = ["amount_ratio", "is_night", "hour_activity_share",
                "new_device_large_amount", "far_from_usual"]

X = data[base_features + new_features]
y = data["target"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Признаков:", X.shape[1])
print("Операций для обучения:", len(X_train))
print("Операций для сравнения:", len(X_valid))

Признаков: 12
Операций для обучения: 3750
Операций для сравнения: 1250


`stratify=y` сохраняет примерно ту же долю фрода в обеих частях.
`random_state=42` позволяет повторить разделение.

## 4. Обучаем модели

Для каждой модели повторяем три знакомых действия:

1. **`fit(X_train, y_train)`** — обучить модель.
2. **`predict(X_valid)`** — получить классы 0 / 1.
3. **`predict_proba(X_valid)[:, 1]`** — получить вероятности класса 1, то есть фрода.

Классы нужны для precision, recall и accuracy. Вероятности — для ROC AUC и PR AUC.
`[:, 1]` выбирает второй столбец: первый относится к классу 0, второй — к классу 1.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### 4.1. Логистическая регрессия

Для логистической регрессии приведём признаки к сопоставимому масштабу с помощью `StandardScaler`.
`make_pipeline` объединяет преобразование и модель: при `fit` масштаб рассчитывается **только на train**,
при прогнозе к validation применяется уже найденное преобразование.

In [6]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

pred_logreg = logreg.predict(X_valid)
proba_logreg = logreg.predict_proba(X_valid)[:, 1]

In [7]:
pd.DataFrame({
    "Факт": y_valid,
    "Прогноз": pred_logreg,
    "Вероятность фрода": proba_logreg,
}).head()

,Факт,Прогноз,Вероятность фрода
2261,0,0,0.307232
3657,0,0,0.082956
1051,0,0,0.041313
347,0,0,0.011509
4480,1,1,0.994385


### 4.2. Решающее дерево

Дерево делит операции на группы с помощью условий по признакам.
Ограничим глубину четырьмя уровнями и оставим не меньше 25 операций в листе.

In [8]:
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=25, random_state=42)
tree.fit(X_train, y_train)

pred_tree = tree.predict(X_valid)
proba_tree = tree.predict_proba(X_valid)[:, 1]

### 4.3. Случайный лес

Лес объединяет прогнозы многих деревьев. Здесь обучим 150 деревьев глубиной не больше 6.
Интерфейс остаётся тем же: `fit`, `predict`, `predict_proba`.

In [9]:
forest = RandomForestClassifier(
    n_estimators=150, max_depth=6, min_samples_leaf=10, random_state=42
)
forest.fit(X_train, y_train)

pred_forest = forest.predict(X_valid)
proba_forest = forest.predict_proba(X_valid)[:, 1]

### 4.4. Градиентный бустинг

Бустинг добавляет деревья последовательно: каждое следующее предлагает поправки к текущему прогнозу.
Здесь используем 100 небольших деревьев. Параметры уже заданы — достаточно запустить ячейку.

In [10]:
boosting = GradientBoostingClassifier(
    n_estimators=100, max_depth=2, learning_rate=0.1, random_state=42
)
boosting.fit(X_train, y_train)

pred_boosting = boosting.predict(X_valid)
proba_boosting = boosting.predict_proba(X_valid)[:, 1]

## 5. Считаем метрики и собираем таблицу

| Метрика | На какой вопрос отвечает | Что передаём |
|---|---|---|
| Precision | Среди операций, отмеченных как фрод, сколько действительно мошеннических? | Классы из `predict` |
| Recall | Какую долю всех фродов мы нашли? | Классы из `predict` |
| Accuracy | Какая доля всех прогнозов верна? | Классы из `predict` |
| ROC AUC | Насколько хорошо фроды оказываются выше обычных операций по оценке модели? | Вероятности класса 1 |
| PR AUC (AP) | Насколько высокой остаётся precision по мере роста recall? | Вероятности класса 1 |

Все метрики здесь лежат в диапазоне от 0 до 1; больше — лучше.
Precision, recall и accuracy оцениваем для стандартного решения `predict` с порогом 0,5.
ROC AUC и AP рассчитываются по вероятностям без выбора одного порога.

В столбце **PR AUC (AP)** используем **Average Precision** — ступенчатую оценку площади под PR-кривой.
Это не площадь методом трапеций. В scikit-learn её считает
[`average_precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).

Следующая готовая функция считает одну строку будущей таблицы.
`zero_division=0` позволяет вернуть 0, если у модели не оказалось прогнозов класса 1.

In [11]:
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score, accuracy_score, average_precision_score
)

def metrics_row(name, predictions, probabilities):
    return {
        "Модель": name,
        "Precision": precision_score(y_valid, predictions, zero_division=0),
        "Recall": recall_score(y_valid, predictions, zero_division=0),
        "ROC AUC": roc_auc_score(y_valid, probabilities),
        "Accuracy": accuracy_score(y_valid, predictions),
        "PR AUC (AP)": average_precision_score(y_valid, probabilities),
    }

In [12]:
results = pd.DataFrame([
    metrics_row("Логистическая регрессия", pred_logreg, proba_logreg),
    metrics_row("Решающее дерево", pred_tree, proba_tree),
    metrics_row("Случайный лес", pred_forest, proba_forest),
    metrics_row("Градиентный бустинг", pred_boosting, proba_boosting),
])

results = results.sort_values("PR AUC (AP)", ascending=False).reset_index(drop=True)
results.round(3)

,Модель,Precision,Recall,ROC AUC,Accuracy,PR AUC (AP)
0,Градиентный бустинг,0.776,0.731,0.886,0.934,0.713
1,Случайный лес,0.806,0.754,0.880,0.942,0.687
2,Решающее дерево,0.792,0.667,0.864,0.930,0.635
3,Логистическая регрессия,0.700,0.450,0.854,0.898,0.612


Таблица отсортирована по AP: сверху — модель с наибольшим значением этой метрики на validation.
По другим метрикам порядок может отличаться.

Сравните две верхние строки: какая модель нашла больше фродов по recall, а какая дала меньше ложных тревог
среди своих срабатываний по precision? Почему одной accuracy недостаточно, когда обычных операций намного больше?

Это сравнение моделей на одном разделении данных. После выбора модели её окончательное качество
проверяют на отдельном test, который не использовали для выбора.

Сохраним таблицу для дальнейшего сравнения:

In [13]:
results.to_csv("model_comparison.csv", index=False, encoding="utf-8-sig")
print("Сохранён model_comparison.csv")

Сохранён model_comparison.csv
